# 05: Choropleth Maps

This notebook demonstrates choropleth map creation using siege_utilities:

1. **Simple GeoDataFrame Choropleths** - Using matplotlib/geopandas directly
2. **ChartGenerator Choropleths** - Using the reporting system
3. **Bivariate Choropleths** - Showing two variables simultaneously
4. **Classification Methods** - Quantiles, equal intervals, natural breaks

## Prerequisites

```bash
pip install -e /path/to/siege_utilities
pip install mapclassify  # For classification schemes
```

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')

import siege_utilities as su
from siege_utilities.geo.spatial_data import get_census_boundaries
from siege_utilities.reporting.chart_generator import ChartGenerator

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Initialize logging
su.configure_shared_logging(level="INFO")

# Set random seed for reproducibility
np.random.seed(42)

su.log_info("Imports successful!")

## 1. Load Geographic Data

We'll use California counties as our base geography.

In [ ]:
# Download California counties
counties = get_census_boundaries(year=2020, geographic_level='county', state_fips='06')
ca_counties = counties[counties['statefp'] == '06'].copy()

su.log_info(f"Loaded {len(ca_counties)} California counties")
su.log_info(f"Columns: {list(ca_counties.columns)[:8]}...")

In [ ]:
# Create sample data - simulated population and income
ca_counties['population'] = np.random.randint(10000, 10000000, size=len(ca_counties))
ca_counties['median_income'] = np.random.randint(40000, 150000, size=len(ca_counties))
ca_counties['unemployment_rate'] = np.random.uniform(2.0, 15.0, size=len(ca_counties))

# Calculate area in square miles
ca_counties['area_sq_mi'] = ca_counties['aland'] / 2_589_988
ca_counties['pop_density'] = ca_counties['population'] / ca_counties['area_sq_mi']

su.log_info("Sample data:")
su.log_info(str(ca_counties[['name', 'population', 'median_income', 'unemployment_rate']].head()))

## 2. Simple Choropleth with GeoPandas

The most straightforward way to create a choropleth using GeoDataFrame.plot()

In [ ]:
# Simple choropleth - population
fig, ax = plt.subplots(1, 1, figsize=(10, 12))

ca_counties.plot(
    column='population',
    ax=ax,
    legend=True,
    legend_kwds={'label': 'Population', 'shrink': 0.6},
    cmap='YlOrRd',
    edgecolor='black',
    linewidth=0.5
)

ax.set_title('California Population by County', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Choropleth with different color schemes
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

# Sequential colormap
ca_counties.plot(
    column='median_income',
    ax=axes[0],
    legend=True,
    cmap='Blues',
    edgecolor='gray',
    linewidth=0.3
)
axes[0].set_title('Median Income (Sequential)')
axes[0].axis('off')

# Diverging colormap
ca_counties.plot(
    column='unemployment_rate',
    ax=axes[1],
    legend=True,
    cmap='RdYlGn_r',  # Red = high unemployment
    edgecolor='gray',
    linewidth=0.3
)
axes[1].set_title('Unemployment Rate (Diverging)')
axes[1].axis('off')

# Population density
ca_counties.plot(
    column='pop_density',
    ax=axes[2],
    legend=True,
    cmap='Purples',
    edgecolor='gray',
    linewidth=0.3
)
axes[2].set_title('Population Density')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 3. Classification Schemes

Different ways to bin continuous data for choropleth maps.

In [ ]:
# Try to import mapclassify for advanced classification
try:
    import mapclassify
    HAS_MAPCLASSIFY = True
    su.log_info("mapclassify available - can use advanced classification")
except ImportError:
    HAS_MAPCLASSIFY = False
    su.log_warning("mapclassify not available - using basic classification")

In [ ]:
if HAS_MAPCLASSIFY:
    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    
    schemes = [
        ('quantiles', 'Quantiles (equal count)'),
        ('equal_interval', 'Equal Interval'),
        ('fisher_jenks', 'Fisher-Jenks (natural breaks)'),
        ('percentiles', 'Percentiles')
    ]
    
    for ax, (scheme, title) in zip(axes.flat, schemes):
        ca_counties.plot(
            column='population',
            ax=ax,
            legend=True,
            scheme=scheme,
            k=5,  # 5 classes
            cmap='YlOrRd',
            edgecolor='black',
            linewidth=0.3
        )
        ax.set_title(title)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    su.log_info("Install mapclassify: pip install mapclassify")

## 4. Bivariate Choropleth

Show two variables simultaneously using a 2D color scheme.

In [ ]:
def create_bivariate_choropleth(gdf, var1, var2, title='Bivariate Choropleth', n_classes=3):
    """
    Create a bivariate choropleth map showing two variables.
    
    Args:
        gdf: GeoDataFrame with geometry and data
        var1: Column name for first variable (x-axis of legend)
        var2: Column name for second variable (y-axis of legend)
        title: Map title
        n_classes: Number of classes per variable (default 3 = 9 total colors)
    """
    # Create quantile bins for each variable
    gdf = gdf.copy()
    gdf['var1_class'] = pd.qcut(gdf[var1], n_classes, labels=False, duplicates='drop')
    gdf['var2_class'] = pd.qcut(gdf[var2], n_classes, labels=False, duplicates='drop')
    
    # Create combined class (0-8 for 3x3)
    gdf['bivar_class'] = gdf['var1_class'] + gdf['var2_class'] * n_classes
    
    # Define bivariate color scheme (3x3)
    # Rows = var2 (low to high), Cols = var1 (low to high)
    bivariate_colors = [
        '#e8e8e8', '#ace4e4', '#5ac8c8',  # var2 low
        '#dfb0d6', '#a5add3', '#5698b9',  # var2 mid
        '#be64ac', '#8c62aa', '#3b4994',  # var2 high
    ]
    
    # Create colormap
    cmap = mcolors.ListedColormap(bivariate_colors)
    
    # Create figure with map and legend
    fig, (ax_map, ax_legend) = plt.subplots(
        1, 2, figsize=(14, 10),
        gridspec_kw={'width_ratios': [4, 1]}
    )
    
    # Plot map
    gdf.plot(
        column='bivar_class',
        ax=ax_map,
        cmap=cmap,
        edgecolor='white',
        linewidth=0.5,
        vmin=0,
        vmax=n_classes*n_classes-1
    )
    ax_map.set_title(title, fontsize=14)
    ax_map.axis('off')
    
    # Create legend
    legend_array = np.arange(n_classes*n_classes).reshape(n_classes, n_classes)
    ax_legend.imshow(legend_array, cmap=cmap, origin='lower')
    ax_legend.set_xlabel(f'{var1} \u2192', fontsize=10)
    ax_legend.set_ylabel(f'{var2} \u2192', fontsize=10)
    ax_legend.set_xticks([])
    ax_legend.set_yticks([])
    ax_legend.set_title('Legend', fontsize=10)
    
    plt.tight_layout()
    return fig

In [ ]:
# Create bivariate choropleth: Population vs Median Income
fig = create_bivariate_choropleth(
    ca_counties,
    var1='population',
    var2='median_income',
    title='California: Population vs Median Income by County'
)
plt.show()

In [ ]:
# Create bivariate choropleth: Population Density vs Unemployment
fig = create_bivariate_choropleth(
    ca_counties,
    var1='pop_density',
    var2='unemployment_rate',
    title='California: Population Density vs Unemployment Rate'
)
plt.show()

## 5. Using ChartGenerator

The ChartGenerator class provides choropleth methods designed for integration with the reporting system.

In [ ]:
# Initialize ChartGenerator
chart_gen = ChartGenerator()

# Prepare data for ChartGenerator (expects location column and value column)
chart_data = ca_counties[['name', 'population', 'median_income']].copy()
chart_data = chart_data.rename(columns={'name': 'county'})

su.log_info("ChartGenerator initialized")
su.log_info("Available methods: create_choropleth_map, create_bivariate_choropleth")

In [ ]:
# Check if Folium is available (required for ChartGenerator maps)
try:
    import folium
    HAS_FOLIUM = True
    su.log_info(f"Folium version: {folium.__version__}")
except ImportError:
    HAS_FOLIUM = False
    su.log_warning("Folium not available - ChartGenerator maps will use fallback")

## 6. Saving Maps

Export choropleth maps to various formats.

In [ ]:
# Save a choropleth to PNG
fig, ax = plt.subplots(1, 1, figsize=(12, 14))

ca_counties.plot(
    column='median_income',
    ax=ax,
    legend=True,
    legend_kwds={
        'label': 'Median Income ($)',
        'shrink': 0.6,
        'format': '${x:,.0f}'
    },
    cmap='Greens',
    edgecolor='black',
    linewidth=0.5
)

ax.set_title('California Median Income by County', fontsize=16, fontweight='bold')
ax.axis('off')

# Save to file
output_path = '/tmp/ca_income_choropleth.png'
fig.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='white')
su.log_info(f"Saved to: {output_path}")

plt.show()

## Summary

This notebook demonstrated:

| Technique | Use Case |
|-----------|----------|
| `gdf.plot(column=...)` | Quick choropleth from GeoDataFrame |
| Color schemes (cmaps) | Sequential, diverging, qualitative |
| Classification schemes | Quantiles, equal interval, natural breaks |
| Bivariate choropleth | Show relationship between two variables |
| ChartGenerator | Integration with reporting system |

**Key Color Schemes:**
- Sequential: `'Blues'`, `'Greens'`, `'YlOrRd'`, `'Purples'`
- Diverging: `'RdBu'`, `'RdYlGn'`, `'BrBG'`
- See: https://matplotlib.org/stable/gallery/color/colormap_reference.html